# Data augmentation practice

**Topic:** Computer Vision &nbsp;·&nbsp; **Difficulty:** Hard

## The data (created in the next cell)

- `X_train` `(4000, 16, 16)`, `y_train`: three shape classes drawn near the center (jitter of at most 1 pixel).
- `X_test` `(1200, 16, 16)`: fresh images of the same three shape classes, with centers shifted by up to ±4 pixels. `y_test` is used by the evaluation cell.
- A no-augmentation pixel model typically loses a lot of accuracy on this test set.

## Your task

1. Augment: generate shifted copies of the training images (shift each image and fill the emptied edge with zeros, since `np.roll` would wrap pixels off one edge onto the other) covering roughly the test-time shift range.
2. Train your classifier (CNN in PyTorch, or MLP on the augmented pixels). The reference solution trains an MLP on 16,000 augmented images, which takes seconds on a CPU.
3. Set `predictions` to your 0/1/2 labels for `X_test`.

## Evaluation

Test accuracy must reach the target shown in the evaluation cell.


In [ ]:
import numpy as np

def draw(n, seed, max_shift):
    r = np.random.default_rng(seed)
    X = r.normal(0.12, 0.08, (n, 16, 16)).clip(0, 1).astype(np.float32)
    y = r.integers(0, 3, n)
    for i in range(n):
        cy = 8 + r.integers(-max_shift, max_shift + 1)
        cx = 8 + r.integers(-max_shift, max_shift + 1)
        b = 1.0 - r.uniform(0, 0.2)
        if y[i] == 0:                                   # filled square
            X[i, cy-2:cy+3, cx-2:cx+3] = b
        elif y[i] == 1:                                 # hollow diamond
            for dy in range(-3, 4):
                dx = 3 - abs(dy)
                X[i, cy+dy, cx-dx] = b
                X[i, cy+dy, cx+dx] = b
        else:                                           # plus sign
            X[i, cy-3:cy+4, cx] = b
            X[i, cy, cx-3:cx+4] = b
    return X, y

X_train, y_train = draw(4000, 401, max_shift=1)   # nearly centered
X_test, y_test = draw(1200, 402, max_shift=4)     # shifted!
print(f"train shift <= 1px, test shift <= 4px: mind the gap")


In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.





In [ ]:
# =====================================================================
# EVALUATION: run this cell to compare your predictions to the truth
# =====================================================================
import numpy as np

TARGET_ACCURACY = 0.8

if "predictions" not in dir():
    raise NameError("Define `predictions` in the solution cell before running this cell.")

_pred = np.asarray(predictions).ravel()
_true = np.asarray(y_test).ravel()
if _pred.shape != _true.shape:
    raise ValueError(f"`predictions` has shape {_pred.shape}, expected {_true.shape}.")

accuracy = float((_pred == _true).mean())
print("=" * 52)
print(f"Test accuracy : {accuracy:.4f}")
print(f"Target        : >= {TARGET_ACCURACY:.2f}")
print("-" * 52)
print("Per-class recall:")
for _cls in np.unique(_true):
    _mask = _true == _cls
    print(f"  class {_cls!r:>6} : {float((_pred[_mask] == _cls).mean()):.3f}   ({int(_mask.sum())} test samples)")
print("=" * 52)
if accuracy >= TARGET_ACCURACY:
    print("PASSED, nice work!")
else:
    print("NOT PASSED yet: revisit your model and try again.")


## Solution

Close the train/test gap with augmentation: make shifted copies of every training image by shifting each one by up to the +/-4 pixel range the test set uses and filling the emptied edges with zeros, then train the classifier on the augmented set.

```python
import numpy as np
from sklearn.neural_network import MLPClassifier

def shift_image(img, dy, dx):
    """Shift img by (dy, dx), filling the emptied edge with zeros."""
    out = np.zeros_like(img)
    h, w = img.shape
    ys, ye = max(0, dy), min(h, h + dy)
    xs, xe = max(0, dx), min(w, w + dx)
    out[ys:ye, xs:xe] = img[ys - dy:ye - dy, xs - dx:xe - dx]
    return out


rng = np.random.default_rng(0)
aug_X, aug_y = [X_train], [y_train]
for _ in range(3):
    dy = rng.integers(-4, 5, len(X_train))
    dx = rng.integers(-4, 5, len(X_train))
    shifted = np.stack([shift_image(img, int(a), int(b))
                        for img, a, b in zip(X_train, dy, dx)])
    aug_X.append(shifted)
    aug_y.append(y_train)

Xa = np.concatenate(aug_X).reshape(-1, 256)
ya = np.concatenate(aug_y)

mlp = MLPClassifier(hidden_layer_sizes=(256, 128), max_iter=300, random_state=0)
mlp.fit(Xa, ya)
predictions = mlp.predict(X_test.reshape(len(X_test), -1))
```
